# Myntra Fashion Clothing Dataset Cleaning
This notebook performs data cleaning and preprocessing on the Myntra Fashion Clothing dataset.

In [5]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

## Data Loading
We start by loading the dataset from the CSV file and inspecting the first few rows.

In [6]:
df = pd.read_csv('./Myntra Fasion Clothing.csv')
df.head()

/var/folders/7f/dlpp4c1j37z0hqpqprc0qpy80000gn/T/ipykernel_19374/1632414677.py:1: DtypeWarning: Columns (9) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('./Myntra Fasion Clothing.csv')


,URL,Product_id,BrandName,Category,Individual_category,category_by_Gender,Description,DiscountPrice (in Rs),OriginalPrice (in Rs),DiscountOffer,SizeOption,Ratings,Reviews
0,https://www.myntra.com/jeans/roadster/roadster...,2296012,Roadster,Bottom Wear,jeans,Men,roadster men navy blue slim fit mid rise clean...,824.0,1499.0,45% OFF,"28, 30, 32, 34, 36",3.9,999.0
1,https://www.myntra.com/track-pants/locomotive/...,13780156,LOCOMOTIVE,Bottom Wear,track-pants,Men,locomotive men black white solid slim fit tra...,517.0,1149.0,55% OFF,"S, M, L, XL",4.0,999.0
2,https://www.myntra.com/shirts/roadster/roadste...,11895958,Roadster,Topwear,shirts,Men,roadster men navy white black geometric print...,629.0,1399.0,55% OFF,"38, 40, 42, 44, 46, 48",4.3,999.0
3,https://www.myntra.com/shapewear/zivame/zivame...,4335679,Zivame,Lingerie & Sleep Wear,shapewear,Women,zivame women black saree shapewear zi3023core0...,893.0,1295.0,31% OFF,"S, M, L, XL, XXL",4.2,999.0
4,https://www.myntra.com/tshirts/roadster/roadst...,11690882,Roadster,Western,tshirts,Women,roadster women white solid v neck pure cotton ...,NaN,599.0,35% OFF,"XS, S, M, L, XL",4.2,999.0


## Initial Data Inspection
Checking the dimensions of the dataset and identifying unique values, nulls, and data types.

In [7]:
df.shape

(526564, 13)

In [8]:
print(df.shape)
print(df.duplicated().sum())
tabel = pd.DataFrame({
    'Unique':df.nunique(),
    'Null':df.isna().sum(),
    'Type':df.dtypes.values
})
display(tabel)

(526564, 13)
0


,Unique,Null,Type
URL,526564,0,object
Product_id,526564,0,int64
BrandName,2088,0,object
Category,8,0,object
Individual_category,92,0,object
category_by_Gender,2,0,object
Description,429766,0,object
DiscountPrice (in Rs),4455,193158,float64
OriginalPrice (in Rs),4914,0,float64
DiscountOffer,1418,74306,object


## Data Pruning
This section focuses on handling missing values in the dataset to ensure data quality and completeness.

In [9]:
# df.drop(columns= ['Reviews', 'Ratings'], inplace = True)
df['Reviews'].fillna(0, inplace = True)
df['Ratings'].fillna(0, inplace = True)

/var/folders/7f/dlpp4c1j37z0hqpqprc0qpy80000gn/T/ipykernel_19374/703892738.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Reviews'].fillna(0, inplace = True)
/var/folders/7f/dlpp4c1j37z0hqpqprc0qpy80000gn/T/ipykernel_19374/703892738.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves 

## Discount Processing
In this section, we extract numerical discount values and calculate the actual discount amounts based on whether the offer is a percentage or a flat rate.

In [10]:
df['DiscountOffer'].unique()

array(['45% OFF', '55% OFF', '31% OFF', ..., 'Rs. 334 OFF', 'Rs. 375 OFF',
       'Rs. 283 OFF'], dtype=object)

In [11]:
df['DiscountOffer'].fillna('0 OFF', inplace = True)

/var/folders/7f/dlpp4c1j37z0hqpqprc0qpy80000gn/T/ipykernel_19374/2024690886.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['DiscountOffer'].fillna('0 OFF', inplace = True)


In [12]:
df['extracted_discount_num'] = df['DiscountOffer'].str.extract(r'(\d+\.?\d*)').astype(float)
df['extracted_discount_num']

0         45.0
1         55.0
2         55.0
3         31.0
4         35.0
          ... 
526559     0.0
526560     0.0
526561     0.0
526562     0.0
526563     0.0
Name: extracted_discount_num, Length: 526564, dtype: float64

In [13]:
is_percent = df['DiscountOffer'].str.contains('%', na=False)
is_flat_rs = df['DiscountOffer'].str.contains('Rs', na=False)

In [14]:
df['Calculated_Discount_Amount'] = np.where(
    is_percent,
    (df['OriginalPrice (in Rs)'] * df['extracted_discount_num']) / 100,
    np.where(is_flat_rs, df['extracted_discount_num'], 0)
)

df['Calculated_Discount_Amount']

0         674.55
1         631.95
2         769.45
3         401.45
4         209.65
           ...  
526559      0.00
526560      0.00
526561      0.00
526562      0.00
526563      0.00
Name: Calculated_Discount_Amount, Length: 526564, dtype: float64

## Price and Percentage Correction
We update the 'DiscountPrice' and 'DiscountPercentage' columns based on our manual calculations to ensure data integrity.

In [15]:
df['DiscountPrice (in Rs)'] = df['OriginalPrice (in Rs)'] - df['Calculated_Discount_Amount']

In [16]:
df['DiscountOffer'] = np.where(
    df['OriginalPrice (in Rs)'] > 0,
    (df['Calculated_Discount_Amount'] / df['OriginalPrice (in Rs)']) * 100,
    0
)
df['DiscountOffer'] = df['DiscountOffer'].round(2)

In [17]:
df.rename(columns = {'DiscountOffer':'DiscountPercentage'}, inplace = True)

## Text Categorization Normalization
Standardizing the casing for brand names and categories to avoid duplicate entries due to case sensitivity.

In [18]:
df.head()

,URL,Product_id,BrandName,Category,Individual_category,category_by_Gender,Description,DiscountPrice (in Rs),OriginalPrice (in Rs),DiscountPercentage,SizeOption,Ratings,Reviews,extracted_discount_num,Calculated_Discount_Amount
0,https://www.myntra.com/jeans/roadster/roadster...,2296012,Roadster,Bottom Wear,jeans,Men,roadster men navy blue slim fit mid rise clean...,824.45,1499.0,45.0,"28, 30, 32, 34, 36",3.9,999.0,45.0,674.55
1,https://www.myntra.com/track-pants/locomotive/...,13780156,LOCOMOTIVE,Bottom Wear,track-pants,Men,locomotive men black white solid slim fit tra...,517.05,1149.0,55.0,"S, M, L, XL",4.0,999.0,55.0,631.95
2,https://www.myntra.com/shirts/roadster/roadste...,11895958,Roadster,Topwear,shirts,Men,roadster men navy white black geometric print...,629.55,1399.0,55.0,"38, 40, 42, 44, 46, 48",4.3,999.0,55.0,769.45
3,https://www.myntra.com/shapewear/zivame/zivame...,4335679,Zivame,Lingerie & Sleep Wear,shapewear,Women,zivame women black saree shapewear zi3023core0...,893.55,1295.0,31.0,"S, M, L, XL, XXL",4.2,999.0,31.0,401.45
4,https://www.myntra.com/tshirts/roadster/roadst...,11690882,Roadster,Western,tshirts,Women,roadster women white solid v neck pure cotton ...,389.35,599.0,35.0,"XS, S, M, L, XL",4.2,999.0,35.0,209.65


In [19]:
print(df.shape)
print(df.duplicated().sum())
tabel = pd.DataFrame({
    'Unique':df.nunique(),
    'Null':df.isna().sum(),
    'Type':df.dtypes.values
})
display(tabel)

(526564, 15)
0


,Unique,Null,Type
URL,526564,0,object
Product_id,526564,0,int64
BrandName,2088,0,object
Category,8,0,object
Individual_category,92,0,object
category_by_Gender,2,0,object
Description,429766,0,object
DiscountPrice (in Rs),17881,0,float64
OriginalPrice (in Rs),4914,0,float64
DiscountPercentage,2379,0,float64


In [20]:
df['BrandName'] = df['BrandName'].astype(str).apply(lambda t: t.lower())
df['BrandName'] = df['BrandName'].apply(lambda t: t.capitalize())

df['Category'] = df['Category'].astype(str).apply(lambda t: t.lower())
df['Category'] = df['Category'].apply(lambda t: t.capitalize())

df['Individual_category'] = df['Individual_category'].astype(str).apply(lambda t: t.lower())
df['Individual_category'] = df['Individual_category'].apply(lambda t: t.capitalize())

In [21]:
print(df.shape)
print(df.duplicated().sum())
tabel = pd.DataFrame({
    'Unique':df.nunique(),
    'Null':df.isna().sum(),
    'Type':df.dtypes.values
})
display(tabel)

(526564, 15)
0


,Unique,Null,Type
URL,526564,0,object
Product_id,526564,0,int64
BrandName,2087,0,object
Category,8,0,object
Individual_category,92,0,object
category_by_Gender,2,0,object
Description,429766,0,object
DiscountPrice (in Rs),17881,0,float64
OriginalPrice (in Rs),4914,0,float64
DiscountPercentage,2379,0,float64


In [22]:
df['Individual_category'].unique()

array(['Jeans', 'Track-pants', 'Shirts', 'Shapewear', 'Tshirts', 'Tops',
       'Trousers', 'Tights', 'Kurta-sets', 'Jumpsuit', 'Kurtas', 'Bra',
       'Shorts', 'Dresses', 'Bath-robe', 'Jackets', 'Socks', 'Briefs',
       'Sweatshirts', 'Sarees', 'Trunk', 'Kurtis', 'Skirts',
       'Night-suits', 'Lounge-pants', 'Palazzos', 'Stockings', 'Jeggings',
       'Leggings', 'Shrug', 'Boxers', 'Dupatta', 'Tunics',
       'Innerwear-vests', 'Sweaters', 'Lounge-shorts', 'Thermal-tops',
       'Capris', 'Nightdress', 'Pyjamas', 'Sports-sandals', 'Dungarees',
       'Tracksuits', 'Camisoles', 'Nehru-jackets', 'Blazers',
       'Thermal-bottoms', 'Lounge-tshirts', 'Lehenga-choli', 'Baby-dolls',
       'Coats', 'Thermal-set', 'Saree-blouse', 'Churidar',
       'Dress-material', 'Boots', 'Lingerie-set', 'Sherwani', 'Co-ords',
       'Flats', 'Swimwear', 'Rain-jacket', 'Patiala', 'Salwar',
       'Harem-pants', 'Patiala-and-dupatta', 'Lingerie-accessories',
       'Saree-accessories', 'Suits', 'Dhoti

## Final Data Refinement
In this section, we perform final cleanup steps, including handling missing categorical values and removing temporary columns used for calculations.

In [23]:
df['category_by_Gender'] = df['category_by_Gender'].fillna('Unknown')

columns_to_drop = ['extracted_discount_num', 'Calculated_Discount_Amount']
df.drop(columns=[c for c in columns_to_drop if c in df.columns], inplace=True)

null_summary = df.isnull().sum()
display(null_summary[null_summary > 0])

Series([], dtype: int64)

## Size Option Cleaning
Here we standardize the `SizeOption` column by handling missing values and extracting the number of available sizes as a new numerical feature.

In [24]:
import numpy as np

df['SizeOption'] = df['SizeOption'].fillna('Not Specified').str.strip()

df['SizeCount'] = df['SizeOption'].apply(lambda x: len(x.split(',')) if x != 'Not Specified' else 0)

print("Size Option Cleaning Summary:")
display(df[['SizeOption', 'SizeCount']].head())
print(f"Unique size counts found: {df['SizeCount'].unique()}")

Size Option Cleaning Summary:


,SizeOption,SizeCount
0,"28, 30, 32, 34, 36",5
1,"S, M, L, XL",4
2,"38, 40, 42, 44, 46, 48",6
3,"S, M, L, XL, XXL",5
4,"XS, S, M, L, XL",5


Unique size counts found: [ 5  4  6  8  7  1 15  3 10 11 12  2  9 23 17 14 13 18 16 26 25 19 20 28
 22 21 55 24 30 44 36 35 32 27 51 40 33 80 41 43 31 34 52 53 42 45 39 48
 49 37 46 38 47 56 29]


## Description Normalization
This step cleans the `Description` column by converting text to lowercase and removing redundant whitespace to ensure consistency.

In [25]:
import re

def clean_description(text):
    if pd.isna(text):
        return ""
    # Convert to lowercase
    text = text.lower()
    # Remove extra spaces and newlines
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['Description'] = df['Description'].apply(clean_description)

print("Cleaned Description Sample:")
display(df[['Description']].head())

Cleaned Description Sample:


,Description
0,roadster men navy blue slim fit mid rise clean...
1,locomotive men black white solid slim fit trac...
2,roadster men navy white black geometric printe...
3,zivame women black saree shapewear zi3023core0...
4,roadster women white solid v neck pure cotton ...


## Column Name Standardization
To make the dataset easier to work with in code, we rename the columns by removing units like '(in Rs)' and replacing spaces with underscores.

In [26]:
import re

def clean_column_name(name):
    # Remove '(in Rs)' and other parenthetical info
    name = re.sub(r'\s*\(.*?\)', '', name)
    # Replace spaces and special characters with underscores
    name = re.sub(r'[^a-zA-Z0-0]', '_', name)
    # Remove multiple underscores and strip
    name = re.sub(r'_+', '_', name).strip('_')
    return name

df.columns = [clean_column_name(col) for col in df.columns]

print("Cleaned Column Names:")
print(df.columns.tolist())
display(df.head(2))

Cleaned Column Names:
['URL', 'Product_id', 'BrandName', 'Category', 'Individual_category', 'category_by_Gender', 'Description', 'DiscountPrice', 'OriginalPrice', 'DiscountPercentage', 'SizeOption', 'Ratings', 'Reviews', 'SizeCount']


,URL,Product_id,BrandName,Category,Individual_category,category_by_Gender,Description,DiscountPrice,OriginalPrice,DiscountPercentage,SizeOption,Ratings,Reviews,SizeCount
0,https://www.myntra.com/jeans/roadster/roadster...,2296012,Roadster,Bottom wear,Jeans,Men,roadster men navy blue slim fit mid rise clean...,824.45,1499.0,45.0,"28, 30, 32, 34, 36",3.9,999.0,5
1,https://www.myntra.com/track-pants/locomotive/...,13780156,Locomotive,Bottom wear,Track-pants,Men,locomotive men black white solid slim fit trac...,517.05,1149.0,55.0,"S, M, L, XL",4.0,999.0,4


## Export Cleaned Dataset
Finally, we save the fully processed and cleaned dataframe to a CSV file for future use.

In [27]:
df.to_csv('./CleanedData.csv', index=False)
print("Refined dataset saved as CleanedData_v2.csv")
display(df.head())

Refined dataset saved as CleanedData_v2.csv


,URL,Product_id,BrandName,Category,Individual_category,category_by_Gender,Description,DiscountPrice,OriginalPrice,DiscountPercentage,SizeOption,Ratings,Reviews,SizeCount
0,https://www.myntra.com/jeans/roadster/roadster...,2296012,Roadster,Bottom wear,Jeans,Men,roadster men navy blue slim fit mid rise clean...,824.45,1499.0,45.0,"28, 30, 32, 34, 36",3.9,999.0,5
1,https://www.myntra.com/track-pants/locomotive/...,13780156,Locomotive,Bottom wear,Track-pants,Men,locomotive men black white solid slim fit trac...,517.05,1149.0,55.0,"S, M, L, XL",4.0,999.0,4
2,https://www.myntra.com/shirts/roadster/roadste...,11895958,Roadster,Topwear,Shirts,Men,roadster men navy white black geometric printe...,629.55,1399.0,55.0,"38, 40, 42, 44, 46, 48",4.3,999.0,6
3,https://www.myntra.com/shapewear/zivame/zivame...,4335679,Zivame,Lingerie & sleep wear,Shapewear,Women,zivame women black saree shapewear zi3023core0...,893.55,1295.0,31.0,"S, M, L, XL, XXL",4.2,999.0,5
4,https://www.myntra.com/tshirts/roadster/roadst...,11690882,Roadster,Western,Tshirts,Women,roadster women white solid v neck pure cotton ...,389.35,599.0,35.0,"XS, S, M, L, XL",4.2,999.0,5
